# 💧 USGS Water Level Prediction – Google Colab Training Demo

**Version:** 1.0 (Training Pipeline Only)  
**Model:** EfficientNet Regression (adapted from production pipeline)  
**Target environment:** Google Colab Free Tier · T4 GPU  

---

## 📁 Expected Folder Structure
```
Water-prediction-colab-version/
├── app.py
├── train_demo.py
├── data_acquisition.py
├── Water_Level_Training_Demo.ipynb
└── water_level_demo/
    ├── data/
    │   ├── images/      ← downloaded HiVIS images
    │   └── labels.csv   ← generated labels
    └── results/
        ├── training/    ← training outputs
        └── inference/   ← inference outputs
```

## 🗂️ CSV Format
Your CSV needs **at least two columns**:
| Column purpose | Accepted names |
|---|---|
| Image filename/path | `image_path`, `dfile_path`, `filename`, `file`, `image` |
| Water level value | `water_level`, `usgstrue_wl`, `gage_height`, `target`, `label` |
| Timestamp (optional) | `dt_pdatetime`, `timestamp`, `datetime`, `date_time` |

## 🚀 Quick Start
1. **Run Cell 0** – checks GPU and sets up folder structure  
2. **Run Cell 1** – installs packages (takes ~2 min on first run)  
3. **Run Cell 2** – verifies repo files and imports the modules from the cloned repo root  
4. **Run Cell 3** – launches the Gradio UI (opens a public share link)  
5. Use the Gradio UI to upload data, configure, and train!

## ⚠️ If Colab Disconnects
- Re-run all cells from the top (Cells 0 → 3).  
- Colab runtime storage is temporary. Download results with Cell 6 when training finishes.

## 💾 Memory Tips (T4 · 15 GB VRAM)
| Config | VRAM usage | Speed |
|---|---|---|
| B3 backbone · 384px · batch 4 | ~4–6 GB ✅ | ~30s/epoch/100img |
| B3 backbone · 512px · batch 4 | ~7–9 GB ⚠️ | ~45s/epoch/100img |
| L2 backbone · 512px · batch 8 | >15 GB ❌ OOM | — |

> If you see **CUDA Out of Memory**: reduce batch size to 2 or image size to 224.

In [1]:
# ╔══════════════════════════════════════════════════════════════╗
# ║  CELL 0 – GPU Check & Folder Setup                         ║
# ║  Run this FIRST every time you start or resume a session.  ║
# ╚══════════════════════════════════════════════════════════════╝

import os, sys, subprocess
import torch

print('=' * 60)
print('  USGS Water Level Prediction – Colab Training Demo')
print('=' * 60)

# ── GPU check ──────────────────────────────────────────────────
print('\n🖥️  GPU Status:')
if torch.cuda.is_available():
    gpu_name = torch.cuda.get_device_name(0)
    total_mem = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f'   ✅ GPU: {gpu_name}')
    print(f'   ✅ VRAM: {total_mem:.1f} GB')
    # Detailed memory summary
    print(f'   ✅ CUDA version: {torch.version.cuda}')
else:
    print('   ⚠️  No GPU detected!')
    print('   → Go to Runtime > Change runtime type > GPU > T4')

print(f'\n🐍 Python: {sys.version.split()[0]}')
print(f'🔦 PyTorch: {torch.__version__}')

# ── Folder structure ───────────────────────────────────────────
REPO_DIR = os.getcwd()
DEFAULT_BASE_DIR = '/content/water_level_demo' if os.path.exists('/content') else os.path.join(REPO_DIR, 'water_level_demo')
BASE_DIR = os.environ.get('WATER_LEVEL_DEMO_BASE_DIR', DEFAULT_BASE_DIR)
os.environ['WATER_LEVEL_DEMO_BASE_DIR'] = BASE_DIR
DIRS = [
    BASE_DIR,
    f'{BASE_DIR}/data',
    f'{BASE_DIR}/data/images',
    f'{BASE_DIR}/results',
    f'{BASE_DIR}/results/training',
    f'{BASE_DIR}/results/inference',
]
for d in DIRS:
    os.makedirs(d, exist_ok=True)

print(f'\n📁 Repo root: {REPO_DIR}')
print(f'📁 Demo data/results directory: {BASE_DIR}')

In [2]:
# ╔══════════════════════════════════════════════════════════════╗
# ║  CELL 1 – Install & Verify Dependencies                    ║
# ║  This only needs to run once per Colab session.            ║
# ╚══════════════════════════════════════════════════════════════╝

import subprocess, sys

# Packages to install (torch / torchvision are pre-installed on Colab)
PACKAGES = [
    'timm>=0.9.0',
    'gradio>=4.0.0',
    'pandas>=1.5.0',
    'numpy>=1.23.0',
    'scikit-learn>=1.1.0',
    'matplotlib>=3.6.0',
    'Pillow>=9.0.0',
    'requests>=2.28.0',
    'tqdm>=4.64.0',
]

print('📦 Installing packages ...')
for pkg in PACKAGES:
    result = subprocess.run(
        [sys.executable, '-m', 'pip', 'install', '-q', pkg],
        capture_output=True, text=True
    )
    status = '✅' if result.returncode == 0 else '❌'
    print(f'  {status} {pkg}')

# Verify key imports
print('\n🔍 Verifying imports ...')
import_checks = [
    ('torch',             'torch'),
    ('torchvision',       'torchvision'),
    ('timm',              'timm'),
    ('gradio',            'gradio'),
    ('sklearn',           'sklearn'),
    ('matplotlib',        'matplotlib'),
    ('PIL',               'PIL'),
]
all_ok = True
for display_name, mod_name in import_checks:
    try:
        mod = __import__(mod_name)
        ver = getattr(mod, '__version__', 'unknown')
        print(f'  ✅ {display_name}: {ver}')
    except ImportError:
        print(f'  ❌ {display_name}: NOT FOUND – please install manually')
        all_ok = False

print()
if all_ok:
    print('✅ All dependencies are ready!')
else:
    print('⚠️  Some packages are missing. Check the errors above.')

In [8]:
import os, sys, importlib

REPO_DIR = os.getcwd()
DEFAULT_BASE_DIR = '/content/water_level_demo' if os.path.exists('/content') else os.path.join(REPO_DIR, 'water_level_demo')
BASE_DIR = os.environ.get('WATER_LEVEL_DEMO_BASE_DIR', DEFAULT_BASE_DIR)
os.environ['WATER_LEVEL_DEMO_BASE_DIR'] = BASE_DIR
REQUIRED_FILES = ['app.py', 'train_demo.py', 'data_acquisition.py']

missing = [name for name in REQUIRED_FILES if not os.path.isfile(os.path.join(REPO_DIR, name))]
if missing:
    raise FileNotFoundError(
        'Run this notebook from the cloned repo root. Missing: ' + ', '.join(missing)
    )

if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)

print(f'📂 Loading modules from repo root: {REPO_DIR}')

for mod_name in ['data_acquisition', 'train_demo', 'app']:
    if mod_name in sys.modules:
        importlib.reload(sys.modules[mod_name])

import data_acquisition, train_demo, app as gradio_app
importlib.reload(data_acquisition); importlib.reload(train_demo); importlib.reload(gradio_app)

print('\n✅ All repo modules loaded — now run Cell 3!')


In [ ]:
# ╔══════════════════════════════════════════════════════════════╗
# ║  CELL 3 – Launch Gradio Training UI                        ║
# ║                                                            ║
# ║  This launches the Gradio web interface.                   ║
# ║  A public share link will appear below – click it to open ║
# ║  the demo in your browser.                                 ║
# ╚══════════════════════════════════════════════════════════════╝

import os, sys
import importlib

REPO_DIR = os.getcwd()
DEFAULT_BASE_DIR = '/content/water_level_demo' if os.path.exists('/content') else os.path.join(REPO_DIR, 'water_level_demo')
BASE_DIR = os.environ.get('WATER_LEVEL_DEMO_BASE_DIR', DEFAULT_BASE_DIR)
os.environ['WATER_LEVEL_DEMO_BASE_DIR'] = BASE_DIR
if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)

# Reload modules in case code was updated
import train_demo
importlib.reload(train_demo)

import app as gradio_app
importlib.reload(gradio_app)

# GPU quick check
train_demo.check_gpu()

print('\n🌐 Launching Gradio UI ...')
print('   (share=True → a public link will appear below)')
print('   The link is valid for 72 hours.')
print('   Gradio debug/show_error is enabled so inference tracebacks print below.')
print()

# Launch the Gradio app
# share=True creates a public tunnelled URL (required for Colab)
demo = gradio_app.launch_gradio(share=True, debug=True, show_error=True)

In [ ]:
# ╔══════════════════════════════════════════════════════════════╗
# ║  CELL 4 – (Optional) Run Training Directly from Python     ║
# ║                                                            ║
# ║  Use this if you prefer running training without the UI,   ║
# ║  or for debugging / scripted experiments.                  ║
# ╚══════════════════════════════════════════════════════════════╝

import sys, os
REPO_DIR = os.getcwd()
DEFAULT_BASE_DIR = '/content/water_level_demo' if os.path.exists('/content') else os.path.join(REPO_DIR, 'water_level_demo')
BASE_DIR = os.environ.get('WATER_LEVEL_DEMO_BASE_DIR', DEFAULT_BASE_DIR)
os.environ['WATER_LEVEL_DEMO_BASE_DIR'] = BASE_DIR
if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)

import train_demo as td
import pandas as pd

# ── Configuration – adjust these ─────────────────────────────
CSV_PATH   = f'{BASE_DIR}/data/labels.csv'         # path to your CSV
IMAGE_DIR  = f'{BASE_DIR}/data/images'             # path to your images folder
RESULTS    = f'{BASE_DIR}/results'
TRAINING_RESULTS = f'{RESULTS}/training'
os.makedirs(TRAINING_RESULTS, exist_ok=True)

# Pinewood Road ROI (x1, y1, x2, y2)
ROI = (951, 0, 1136, 1920)

# ── Load and prepare data ────────────────────────────────────
df = pd.read_csv(CSV_PATH)
img_col, target_col, time_col = td.detect_columns(df)
print(f'Columns: image={img_col}, target={target_col}, time={time_col}')

mappings = td.build_image_label_mapping(
    df, img_col, target_col,
    image_dir=IMAGE_DIR,
    roi=None,
    max_images=1000,
    seed=42,
)

# ── Train ────────────────────────────────────────────────────
results = td.train_model(
    mappings         = mappings,
    roi              = ROI,
    results_dir      = TRAINING_RESULTS,
    num_epochs       = 5,
    batch_size       = 4,
    input_img_size   = 384,
    learning_rate    = 2e-4,
    val_ratio        = 0.15,
    test_ratio       = 0.10,
    param_freeze_ratio = 0.7,
    seed             = 42,
    backbone_name    = 'tf_efficientnet_b3.ns_jft_in1k',
    save_to_drive    = False,
)

print('\nTraining complete!')
print('Best val loss:', results['best_val_loss'])
print('Outputs:', results)

In [ ]:
# ╔══════════════════════════════════════════════════════════════╗
# ║  CELL 5 – View Training Results                            ║
# ╚══════════════════════════════════════════════════════════════╝

import os, json, glob, pandas as pd
from IPython.display import display, Image as IPImage
import matplotlib.pyplot as plt

REPO_DIR = os.getcwd()
DEFAULT_BASE_DIR = '/content/water_level_demo' if os.path.exists('/content') else os.path.join(REPO_DIR, 'water_level_demo')
BASE_DIR = os.environ.get('WATER_LEVEL_DEMO_BASE_DIR', DEFAULT_BASE_DIR)
RESULTS = os.path.join(BASE_DIR, 'results')
TRAINING_RESULTS = os.path.join(RESULTS, 'training')

def latest(pattern):
    matches = glob.glob(os.path.join(TRAINING_RESULTS, pattern))
    return max(matches, key=os.path.getmtime) if matches else None

# ── List output files ─────────────────────────────────────────
print('📁 Files in training results directory:')
if os.path.exists(TRAINING_RESULTS):
    for f in sorted(os.listdir(TRAINING_RESULTS)):
        size = os.path.getsize(os.path.join(TRAINING_RESULTS, f))
        print(f'  {f:40s}  {size:>10,} bytes')
else:
    print('  (empty – run training first)')

# ── Show config ───────────────────────────────────────────────
config_path = latest('config_*.json')
if config_path and os.path.exists(config_path):
    print('\n📄 Training Config:')
    with open(config_path) as f:
        cfg = json.load(f)
    for k, v in cfg.items():
        print(f'  {k:25s}: {v}')

# ── Show loss history ─────────────────────────────────────────
history_path = latest('training_history_*.csv')
if history_path and os.path.exists(history_path):
    print('\n📊 Loss History:')
    df = pd.read_csv(history_path)
    display(df)

# ── Display loss plot ─────────────────────────────────────────
plot_path = latest('training_loss_plot_*.png')
if plot_path and os.path.exists(plot_path):
    print('\n📈 Loss Plot:')
    display(IPImage(plot_path))

In [ ]:
# ╔══════════════════════════════════════════════════════════════╗
# ║  CELL 6 – Download Results ZIP                             ║
# ║                                                            ║
# ║  Creates a ZIP of all training outputs and downloads it.   ║
# ╚══════════════════════════════════════════════════════════════╝

import os, shutil
from google.colab import files

REPO_DIR = os.getcwd()
DEFAULT_BASE_DIR = '/content/water_level_demo' if os.path.exists('/content') else os.path.join(REPO_DIR, 'water_level_demo')
BASE_DIR = os.environ.get('WATER_LEVEL_DEMO_BASE_DIR', DEFAULT_BASE_DIR)
RESULTS   = os.path.join(BASE_DIR, 'results')
TRAINING_RESULTS = os.path.join(RESULTS, 'training')
ZIP_PATH  = os.path.join(os.getcwd(), 'water_level_training_outputs.zip')

if os.path.exists(TRAINING_RESULTS) and os.listdir(TRAINING_RESULTS):
    shutil.make_archive(
        base_name = ZIP_PATH.replace('.zip', ''),
        format    = 'zip',
        root_dir  = TRAINING_RESULTS,
    )
    print(f'✅ Created: {ZIP_PATH}')
    files.download(ZIP_PATH)
else:
    print('⚠️  No results to download. Run training first.')